<a href="https://colab.research.google.com/github/amalvaz-ai/Capstone-Analysis/blob/main/Net_Sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup

!pip install snscrape
!pip install requests beautifulsoup4 feedparser nltk

import nltk
nltk.download('vader_lexicon')
import feedparser
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()
import urllib.parse
import json
import subprocess
from typing import List, Optional, Tuple
# --- Sentiment (VADER) ---
import time
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.1 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=8830e7a7db721fde2988b8b478233677741f4b466fbe4386f2a464b2cbf59d08
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [ ]:
def reddit_sentiment(ticker):
    url = f"https://api.pullpush.io/reddit/search/comment/?q={ticker}&size=50"
    try:
        data = requests.get(url, timeout=10).json()
        comments = [c["body"] for c in data.get("data", [])]
        if not comments:
            return 0.0
        scores = [sia.polarity_scores(c)["compound"] for c in comments]
        return sum(scores) / len(scores)
    except:
        return 0.0

In [ ]:
def finviz_sentiment(ticker):
    url = f"https://finviz.com/quote.ashx?t={ticker}"
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        html = requests.get(url, headers=headers, timeout=10).text
        soup = BeautifulSoup(html, "html.parser")
        news_table = soup.find("table", class_="fullview-news-outer")
        rows = news_table.find_all("tr")[:20]

        texts = [r.text.strip() for r in rows]
        scores = [sia.polarity_scores(t)["compound"] for t in texts]
        return sum(scores) / len(scores)
    except:
        return 0.0

In [ ]:
COMPANY_NAMES = {
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "TSLA": "Tesla"

}

def rss_sentiment(ticker):
    name = COMPANY_NAMES.get(ticker.upper(), "")

    # Build query: "AAPL OR Apple"
    if name:
        query = f"{ticker} OR {name}"
    else:
        query = ticker

    # URL encode it safely
    encoded_query = urllib.parse.quote(query)

    url = f"https://news.google.com/rss/search?q={encoded_query}"

    try:
        feed = feedparser.parse(url)
        texts = []

        for entry in feed.entries[:20]:
            text = entry.title + " " + entry.get("summary", "")
            texts.append(text)

        if not texts:
            return 0.0

        scores = [sia.polarity_scores(t)["compound"] for t in texts]
        return sum(scores) / len(scores)

    except Exception as e:
        print("RSS error:", e)
        return 0.0

In [ ]:
# Ensure VADER lexicon exists (safe to call; downloads only if missing)
try:
    nltk.data.find("sentiment/vader_lexicon.zip")
except LookupError:
    nltk.download("vader_lexicon")

sia = SentimentIntensityAnalyzer()

# --- Nitter mirrors (best-effort; mirrors may go up/down) ---
NITTER_INSTANCES = [
    "https://nitter.poast.org",
    "https://nitter.cz",
    "https://nitter.uni-sonia.com",
    "https://nitter.esmailelbob.xyz",
]

# Try multiple selectors because Nitter markup differs by instance/version
NITTER_SELECTORS = [
    "div.tweet-content",
    "div.tweet-body",
    "p.tweet-content",
    "article .tweet-content",
    "article .tweet-text",
    "div.main-tweet > div.tweet-content",  # fixed: '>' not '&gt;'
]

DEFAULT_HEADERS = {"User-Agent": "Mozilla/5.0"}


def _normalize_symbol(symbol: str) -> Tuple[str, str]:
    """
    Accepts 'AAPL' or '$AAPL' and returns ('AAPL', '$AAPL')
    """
    ticker = (symbol or "").strip().lstrip("$").upper()
    cashtag = f"${ticker}" if ticker else ""
    return ticker, cashtag


def _dedupe_keep_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def _avg_compound(texts: List[str]) -> float:
    if not texts:
        return 0.0
    scores = [sia.polarity_scores(t)["compound"] for t in texts]
    return sum(scores) / len(scores)


def _fetch_from_nitter(
    cashtag: str,
    max_tweets: int = 50,
    timeout: int = 10,
    headers: Optional[dict] = None,
) -> Optional[float]:
    """
    Scrape cashtag search results from Nitter instances.
    Returns average VADER compound score, or None if nothing found.
    """
    if not cashtag:
        return None

    headers = headers or DEFAULT_HEADERS

    # Nitter search expects URL-encoded query.
    # '$' is %24; but cashtag might include $ already, so encode explicitly.
    # We'll just replace leading '$' with '%24' and keep rest.
    # Example: '$AAPL' -> '%24AAPL'
    q = "%24" + cashtag.lstrip("$")

    for base in NITTER_INSTANCES:
        try:
            # fixed: '&' not '&amp;'
            url = f"{base}/search?f=tweets&q={q}"
            resp = requests.get(url, headers=headers, timeout=timeout)

            if resp.status_code != 200 or not resp.text:
                continue

            soup = BeautifulSoup(resp.text, "html.parser")

            tweets = []
            for sel in NITTER_SELECTORS:
                for node in soup.select(sel):
                    text = node.get_text(" ", strip=True)
                    if text:
                        tweets.append(text)

            tweets = _dedupe_keep_order(tweets)[:max_tweets]
            if tweets:
                return _avg_compound(tweets)

        except Exception:
            # Best effort: try next instance
            continue

    return None


def _fetch_from_snscrape(
    cashtag: str,
    max_tweets: int = 200,
    since: str = "2024-01-01",
    timeout: int = 30,
) -> Optional[float]:
    """
    Uses snscrape CLI (must be installed and accessible in PATH).
    Returns average VADER compound score, or None if nothing found.
    """
    if not cashtag:
        return None

    # Build query explicitly (cashtag + date filter)
    query = f"{cashtag} since:{since}"

    # Use list args (shell=False) to avoid shell expansion issues with '$'
    cmd = ["snscrape", "--jsonl", "twitter-search", query]

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout,
            check=False,  # don't raise; we'll interpret output
        )

        # If snscrape errored and returned nothing, treat as failure
        if result.returncode != 0 and not (result.stdout or "").strip():
            return None

        tweets = []
        for line in (result.stdout or "").splitlines():
            if len(tweets) >= max_tweets:
                break
            try:
                obj = json.loads(line)
                content = obj.get("content") or ""
                if content:
                    tweets.append(content)
            except json.JSONDecodeError:
                continue

        tweets = _dedupe_keep_order(tweets)
        if tweets:
            return _avg_compound(tweets)

    except Exception:
        return None


def twitter_sentiment(ticker: str) -> float:
    """
    Aggregates sentiment from various Twitter-like sources.
    Prioritizes Nitter, then falls back to snscrape.
    """
    _, cashtag = _normalize_symbol(ticker)

    # Try Nitter first
    nitter_score = _fetch_from_nitter(cashtag)
    if nitter_score is not None:
        return nitter_score

    # Fallback to snscrape
    snscrape_score = _fetch_from_snscrape(cashtag)
    if snscrape_score is not None:
        return snscrape_score

    return 0.0

## GitHub Sentiment Agent - Fetch Data




In [ ]:
def fetch_github_text(ticker):
    texts = []
    headers = {'User-Agent': 'Colab-Financial-Agent'} # User-Agent is required by GitHub API

    # 1. Search Repositories
    try:
        url_repos = f"https://api.github.com/search/repositories?q={ticker}&per_page=5"
        response = requests.get(url_repos, headers=headers)
        if response.status_code == 200:
            items = response.json().get('items', [])
            for item in items:
                desc = item.get('description')
                if desc:
                    texts.append(desc)
        else:
            print(f"Warning: Failed to search repos for {ticker} (Status: {response.status_code})")
    except Exception as e:
        print(f"Error searching repos for {ticker}: {e}")

    time.sleep(1) # Pause to respect unauthenticated rate limits

    # 2. Search Issues
    try:
        url_issues = f"https://api.github.com/search/issues?q={ticker}&per_page=5"
        response = requests.get(url_issues, headers=headers)
        if response.status_code == 200:
            items = response.json().get('items', [])
            for item in items:
                title = item.get('title', '')
                body = item.get('body', '')
                # Handle None types if body is missing
                if body is None:
                    body = ""
                combined_text = f"{title}. {body}"
                if combined_text.strip():
                    texts.append(combined_text)
        else:
             print(f"Warning: Failed to search issues for {ticker} (Status: {response.status_code})")
    except Exception as e:
        print(f"Error searching issues for {ticker}: {e}")

    time.sleep(1) # Pause again
    return texts

# Main execution
# Override tickers to AAPL as requested
tickers = ['AAPL', 'MSFT', 'TSLA']

github_data = {}
print(f"Fetching GitHub data for tickers: {tickers}...")

for ticker in tickers:
    print(f"Processing {ticker}...")
    github_data[ticker] = fetch_github_text(ticker)
    print(f"Found {len(github_data[ticker])} documents for {ticker}.")

# Display sample
if tickers and github_data[tickers[0]]:
    print(f"\nSample text for {tickers[0]}:")
    print(github_data[tickers[0]][0][:200] + "...")
else:
    print("\nNo data found or no tickers available.")

Fetching GitHub data for tickers: ['AAPL', 'MSFT', 'TSLA']...
Processing AAPL...
Found 10 documents for AAPL.
Processing MSFT...
Found 9 documents for MSFT.
Processing TSLA...
Found 10 documents for TSLA.

Sample text for AAPL:
Customize video player base on AVPlayer...


## GitHub Sentiment Agent - Analyze Sentiment



In [ ]:


# Download VADER lexicon
nltk.download('vader_lexicon')

# Initialize VADER analyzer
sia = SentimentIntensityAnalyzer()

sentiment_results = {}

print("Analyzing sentiment...")

for ticker, texts in github_data.items():
    if texts:
        # Calculate compound score for each text
        scores = [sia.polarity_scores(text)['compound'] for text in texts]
        # Compute average score and scale it from -1 to 1 to -100 to 100
        avg_score_scaled = np.mean(scores) * 100

        # Determine layman's terms
        if avg_score_scaled > 50:
            interpretation = "Very Positive"
        elif avg_score_scaled > 20:
            interpretation = "Positive"
        elif avg_score_scaled >= -20 and avg_score_scaled <= 20:
            interpretation = "Neutral"
        elif avg_score_scaled < -20 and avg_score_scaled >= -50:
            interpretation = "Negative"
        else:
            interpretation = "Very Negative"

        sentiment_results[ticker] = {
            'score': avg_score_scaled,
            'interpretation': interpretation
        }
    else:
        sentiment_results[ticker] = {
            'score': 0.0,
            'interpretation': 'Neutral (No data)'
        }

print("\nSentiment Results (scaled -100 to 100 with interpretation):")
for ticker, result in sentiment_results.items():
    print(f"{ticker}: Score = {result['score']:.2f}, Interpretation = {result['interpretation']}")

Analyzing sentiment...

Sentiment Results (scaled -100 to 100 with interpretation):
AAPL: Score = 12.18, Interpretation = Neutral
MSFT: Score = 18.59, Interpretation = Neutral
TSLA: Score = 6.25, Interpretation = Neutral


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [ ]:

# Download VADER lexicon silently to avoid stderr output
nltk.download('vader_lexicon', quiet=True)

# Initialize VADER analyzer
sia = SentimentIntensityAnalyzer()

sentiment_results = {}

print("Analyzing sentiment for updated tickers...")

for ticker, texts in github_data.items():
    if texts:
        # Calculate compound score for each text
        scores = [sia.polarity_scores(text)['compound'] for text in texts]
        # Compute average score
        avg_score = np.mean(scores)
        sentiment_results[ticker] = avg_score
    else:
        sentiment_results[ticker] = 0.0

print("\nSentiment Results:")
print(sentiment_results)

Analyzing sentiment for updated tickers...

Sentiment Results:
{'AAPL': np.float64(0.12177), 'MSFT': np.float64(0.18586666666666668), 'TSLA': np.float64(0.062490000000000004)}


In [ ]:
def combined_sentiment(ticker):
    sources = {
        "reddit": reddit_sentiment(ticker),
        "finviz": finviz_sentiment(ticker),
        "rss": rss_sentiment(ticker),
        "twitter": twitter_sentiment(ticker),
        "github": sentiment_results.get(ticker, 0.0) # Add GitHub sentiment
    }

    # Filter out sources with 0.0 score
    non_zero_scores = [score for score in sources.values() if score != 0.0]

    if not non_zero_scores:
        final_score = 0.0  # If all scores are 0.0 or no sources, return 0.0
    else:
        final_score = sum(non_zero_scores) / len(non_zero_scores)

    return final_score, sources

In [ ]:
tickers = ["AAPL", "MSFT", "TSLA"]

for t in tickers:
    final, breakdown = combined_sentiment(t)
    print(f"\n=== {t} ===")
    print("Breakdown:", breakdown)
    print("Final Sentiment:", final)


=== AAPL ===
Breakdown: {'reddit': 0.430612, 'finviz': -0.03165, 'rss': 0.112975, 'twitter': 0.0, 'github': np.float64(0.12177)}
Final Sentiment: 0.15842675

=== MSFT ===
Breakdown: {'reddit': 0.223296, 'finviz': -0.116285, 'rss': 0.18856, 'twitter': 0.0, 'github': np.float64(0.18586666666666668)}
Final Sentiment: 0.12035941666666668

=== TSLA ===
Breakdown: {'reddit': 0.32583599999999996, 'finviz': 0.01616, 'rss': 0.012859999999999988, 'twitter': 0.0, 'github': np.float64(0.062490000000000004)}
Final Sentiment: 0.10433649999999998


In [ ]:
def interpret_sentiment(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"


tickers = ["AAPL", "MSFT", "TSLA"]

for t in tickers:
    final, breakdown = combined_sentiment(t)
    print(f"\n=== {t} ===")
    interpreted_breakdown = {source: interpret_sentiment(score) for source, score in breakdown.items()}
    print("Breakdown (interpreted):", interpreted_breakdown)
    print("Final Sentiment (interpreted):", interpret_sentiment(final))
    print("Breakdown (raw):", breakdown)
    print("Final Sentiment (raw):", final)


=== AAPL ===
Breakdown (interpreted): {'reddit': 'positive', 'finviz': 'neutral', 'rss': 'positive', 'twitter': 'neutral', 'github': 'positive'}
Final Sentiment (interpreted): positive
Breakdown (raw): {'reddit': 0.430612, 'finviz': -0.03165, 'rss': 0.112975, 'twitter': 0.0, 'github': np.float64(0.12177)}
Final Sentiment (raw): 0.15842675

=== MSFT ===
Breakdown (interpreted): {'reddit': 'positive', 'finviz': 'negative', 'rss': 'positive', 'twitter': 'neutral', 'github': 'positive'}
Final Sentiment (interpreted): positive
Breakdown (raw): {'reddit': 0.223296, 'finviz': -0.116285, 'rss': 0.18856, 'twitter': 0.0, 'github': np.float64(0.18586666666666668)}
Final Sentiment (raw): 0.12035941666666668

=== TSLA ===
Breakdown (interpreted): {'reddit': 'positive', 'finviz': 'neutral', 'rss': 'neutral', 'twitter': 'neutral', 'github': 'positive'}
Final Sentiment (interpreted): positive
Breakdown (raw): {'reddit': 0.32583599999999996, 'finviz': 0.01616, 'rss': 0.012859999999999988, 'twitter': 0

## Create General Market Sentiment Function

### Subtask:
Develop a new function, `general_market_sentiment_rss`, that processes articles from the `RSS_FEEDS` without filtering by a specific ticker to get an overall sentiment score. This function will use the existing `SentimentIntensityAnalyzer`.


**Reasoning**:
I will define the `general_market_sentiment_rss` function as specified in the instructions, which will parse RSS feeds, extract text, calculate sentiment scores using the existing `SentimentIntensityAnalyzer`, and return the average sentiment.



In [ ]:
def general_market_sentiment_rss():
    texts = []
    for feed in RSS_FEEDS:
        try:
            parsed = feedparser.parse(feed)
            for entry in parsed.entries:
                # Check if title and summary exist before concatenating
                title = entry.get('title', '')
                summary = entry.get('summary', '')
                combined_text = f"{title} {summary}".strip()
                if combined_text:
                    texts.append(combined_text)
        except:
            continue # Continue to the next feed if an error occurs

    if not texts:
        return 0.0

    scores = [sia.polarity_scores(t)["compound"] for t in texts]

    # Filter out scores that are 0.0 before calculating the average
    non_zero_scores = [score for score in scores if score != 0.0]

    if not non_zero_scores:
        return 0.0 # Return 0.0 if all scores were 0.0 or no texts were found
    else:
        return sum(non_zero_scores) / len(non_zero_scores)

**Reasoning**:
Now that the `general_market_sentiment_rss` function is defined, I will call it to calculate and display the overall market sentiment based on the RSS feeds.



In [ ]:
RSS_FEEDS = [
    "https://news.google.com/rss/headlines/section/topic/BUSINESS?hl=en-US&gl=US&ceid=US:en",
    "https://rss.nytimes.com/services/xml/rss/nyt/Economy.xml",
    "https://feeds.a.dj.com/rss/RssFeedStored.aspx?form=MRSSNF&band=NONE",
    "https://www.cnbc.com/id/10000664/device/rss/rss.html"
]

overall_market_sentiment = general_market_sentiment_rss()
print(f"Overall Market Sentiment (RSS): {overall_market_sentiment}")

Overall Market Sentiment (RSS): -0.13982264150943396


## Calculate and Interpret Overall Sentiment

### Subtask:
Call the `general_market_sentiment_rss` function to obtain the overall market sentiment score. Then, interpret this raw score as 'positive', 'negative', or 'neutral' using the `interpret_sentiment` function and display the result.


**Reasoning**:
I will interpret the `overall_market_sentiment` using the `interpret_sentiment` function and then print both the raw and interpreted sentiment scores as requested.



In [ ]:
interpreted_market_sentiment = interpret_sentiment(overall_market_sentiment)
print(f"Overall Market Sentiment (Raw): {overall_market_sentiment}")
print(f"Overall Market Sentiment (Interpreted): {interpreted_market_sentiment}")

Overall Market Sentiment (Raw): -0.13982264150943396
Overall Market Sentiment (Interpreted): negative


## Historic Sentiment Preparation

## Update CSV

In [ ]:
import pandas as pd
from datetime import datetime
from google.colab import drive
import os
import json
from google.colab import auth

# --- Google Drive Authentication ---
# This block handles authentication for Google Drive access.
# It attempts to use a service account key if found for automated access.
# If not, it falls back to interactive OAuth user authentication.

# Placeholder for service account key path
SERVICE_ACCOUNT_KEY_FILE = '/content/drive/MyDrive/client_secret_1032730659429-tuof0tavu9rh4l3q8tldetf3dn842i81.apps.googleusercontent.com' # Adjust this path as needed

# Check if the service account key file exists
if os.path.exists(SERVICE_ACCOUNT_KEY_FILE):
    # Service Account Authentication (for automated, non-interactive use)
    print(f"Service account key file '{SERVICE_ACCOUNT_KEY_FILE}' found. Attempting service account authentication.")
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = SERVICE_ACCOUNT_KEY_FILE
    try:
        # For full Drive API access beyond just mounting, you might need to
        # use google-auth-library or gspread with these credentials.
        # drive.mount() itself relies on the environment variable for specific scenarios
        # or defaults to interactive if not fully configured.
        drive.mount('/content/drive', force_remount=True)
        print("Google Drive mounted using (attempted) service account context.")
    except Exception as e:
        print(f"Warning: Could not non-interactively mount Google Drive with service account. Error: {e}")
        print("File operations to '/content/drive/MyDrive' might fail.")
else:
    # Interactive User Authentication (OAuth)
    print(f"Service account key file '{SERVICE_ACCOUNT_KEY_FILE}' not found. Initiating interactive OAuth authentication.")
    try:
        # Authenticate the user via OAuth 2.0 flow
        auth.authenticate_user()
        print("User authenticated successfully via OAuth.")
        # Mount Google Drive after successful authentication
        drive.mount('/content/drive', force_remount=True)
        print("Google Drive mounted using interactive authentication.")
    except Exception as e:
        print(f"Error during interactive OAuth authentication or mounting Google Drive: {e}. File operations to '/content/drive/MyDrive' will fail.")

output_path = '/content/drive/MyDrive/market_sentiment_data.csv'

# Prepare new data for CSV
csv_data = []
current_date_str = datetime.now().strftime('%Y-%m-%d') # For Date column (YYYY-MM-DD)
current_timestamp_run_number = datetime.now().strftime('%Y%m%d%H%M%S') # For Run_Number (YYYYMMDDHHMMSS)

# Define tickers (previously defined in other cells, but needed here for scope)
tickers = ["AAPL", "MSFT", "TSLA"]

# Ensure tickers and overall_market_sentiment are available (from previous cells)
# overall_market_sentiment = overall_market_sentiment

for t in tickers:
    final_sentiment_for_ticker, _ = combined_sentiment(t)
    csv_data.append({
        "Ticker": t,
        "Date": current_date_str,
        "Sentiment": final_sentiment_for_ticker,
        "Overall_Market_Sentiment": overall_market_sentiment
    })

new_df = pd.DataFrame(csv_data)
# Apply rounding to new_df
new_df['Sentiment'] = new_df['Sentiment'].round(2)
new_df['Overall_Market_Sentiment'] = new_df['Overall_Market_Sentiment'].round(2)

previous_data_changed = False
existing_df = pd.DataFrame() # Initialize empty DataFrame

# Define the expected columns for the CSV file
expected_cols = ['Ticker', 'Date', 'Sentiment', 'Overall_Market_Sentiment', 'Run_Number']

# Check if the CSV file already exists
if os.path.exists(output_path):
    try:
        # Read the CSV, explicitly specifying names and no header
        # This handles cases where the file might have inconsistent column counts or a malformed header
        existing_df = pd.read_csv(output_path, names=expected_cols, header=None)

        # After reading with header=None, the actual header line (if present)
        # will be part of the DataFrame. We need to identify and remove it.
        # A simple check: if the 'Ticker' column in the first row contains the string 'Ticker',
        # it's likely the header that was read as data.
        if not existing_df.empty and existing_df.iloc[0]['Ticker'] == 'Ticker':
            existing_df = existing_df.iloc[1:].reset_index(drop=True)

    except pd.errors.ParserError as e:
        print(f"Warning: Could not parse existing CSV due to error: {e}. Attempting to re-create the file.")
        # If parsing fails, treat it as if the file didn't exist or was empty for this run.
        existing_df = pd.DataFrame(columns=expected_cols)
        # Ensure we write a new, consistent file if there was a parsing error
        previous_data_changed = True

    # Clean up existing_df: Drop rows where 'Run_Number' is NaN
    if 'Run_Number' in existing_df.columns:
        initial_rows = len(existing_df)
        # Ensure 'Run_Number' is numeric before dropping NaNs
        existing_df['Run_Number'] = pd.to_numeric(existing_df['Run_Number'], errors='coerce')
        existing_df.dropna(subset=['Run_Number'], inplace=True)
        if len(existing_df) < initial_rows:
            # If rows were dropped, re-save the cleaned existing_df to the file
            existing_df.to_csv(output_path, index=False)
            print(f"Cleaned existing data: Dropped {initial_rows - len(existing_df)} rows with NaN 'Run_Number'.")

    # Ensure existing_df's numeric columns are also rounded for consistent comparison
    if not existing_df.empty:
        existing_df['Sentiment'] = pd.to_numeric(existing_df['Sentiment'], errors='coerce').round(2)
        existing_df['Overall_Market_Sentiment'] = pd.to_numeric(existing_df['Overall_Market_Sentiment'], errors='coerce').round(2)


    # Determine if there's a change in sentiment compared to the last run
    if not existing_df.empty and 'Run_Number' in existing_df.columns:
        # Convert 'Run_Number' to string for timestamp comparison if necessary, or just pick max
        last_run_id = existing_df['Run_Number'].astype(str).max()
        last_run_df = existing_df[existing_df['Run_Number'].astype(str) == last_run_id]

        cols_to_compare = ['Ticker', 'Sentiment', 'Overall_Market_Sentiment']
        last_run_comparison = last_run_df[cols_to_compare].reset_index(drop=True)
        new_data_comparison = new_df[cols_to_compare].reset_index(drop=True)

        # Sort both DataFrames to ensure consistent comparison regardless of original order
        last_run_comparison_sorted = last_run_comparison.sort_values(by=['Ticker']).reset_index(drop=True)
        new_data_comparison_sorted = new_data_comparison.sort_values(by=['Ticker']).reset_index(drop=True)

        if not last_run_comparison_sorted.equals(new_data_comparison_sorted):
            previous_data_changed = True
    else:
        # No 'Run_Number' column or empty existing_df after cleaning, treat as first run with timestamps
        previous_data_changed = True # Forces append as it's essentially a new dataset structure or first run
else:
    # File does not exist, so it's the first run
    previous_data_changed = True

# Add current_timestamp_run_number to the new DataFrame
new_df['Run_Number'] = current_timestamp_run_number

# Append new data only if sentiment values have changed or it's the first run
if previous_data_changed:
    if os.path.exists(output_path) and not existing_df.empty:
        # Append to existing CSV
        # Ensure 'Run_Number' column is consistent before concat
        new_df['Run_Number'] = new_df['Run_Number'].astype(existing_df['Run_Number'].dtype)
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        combined_df.to_csv(output_path, index=False) # Overwrite with combined data
        df = combined_df # Update df in kernel
        print(f"New market sentiment data (Run {current_timestamp_run_number}) appended to {output_path}")
    else:
        # Create new CSV or if existing_df was empty after cleaning
        new_df.to_csv(output_path, index=False)
        df = new_df # Update df in kernel with the newly created dataframe
        print(f"Market sentiment data (Run {current_timestamp_run_number}) saved to new file {output_path}")
else:
    df = existing_df # If no change, df should still reflect the existing data
    print("No significant change in sentiment values. Data not appended.")

# Display the updated DataFrame (either appended or existing)
display(df)

Service account key file '/content/drive/MyDrive/client_secret_1032730659429-tuof0tavu9rh4l3q8tldetf3dn842i81.apps.googleusercontent.com' not found. Initiating interactive OAuth authentication.
User authenticated successfully via OAuth.
Mounted at /content/drive
Google Drive mounted using interactive authentication.
New market sentiment data (Run 20260322141835) appended to /content/drive/MyDrive/market_sentiment_data.csv


,Ticker,Date,Sentiment,Overall_Market_Sentiment,Run_Number
0,AAPL,2026-02-24,0.20,0.02,1.000000e+00
1,MSFT,2026-02-24,0.11,0.02,1.000000e+00
2,TSLA,2026-02-24,0.06,0.02,1.000000e+00
3,AAPL,2026-02-24,0.19,0.02,2.026022e+13
4,MSFT,2026-02-24,0.13,0.02,2.026022e+13
...,...,...,...,...,...
210,MSFT,2026-03-21,0.14,-0.21,2.026032e+13
211,TSLA,2026-03-21,0.11,-0.21,2.026032e+13
212,AAPL,2026-03-22,0.16,-0.14,2.026032e+13
213,MSFT,2026-03-22,0.12,-0.14,2.026032e+13
